# PCA, MCA, FAMD
개별 자료의 상관관계를 이용해 자료의 차원(dimensionality of data)을 줄이는 통계학 기법(들)

- PCA(Principal Component Analysis): 주성분분석
- MCA(Multiple Correspondence Analysis): 다중 대응분석
- FAMD(Factor Analysis of Mixed Data): 혼합 자료의 요인분석 (feat. 안티그래비티 번역)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.decomposition import PCA # 주성분분석
from sklearn.preprocessing import StandardScaler # 을 하려면 스케일러가 필요합니다

import prince # FAMD용

In [ ]:
# 그래프 기본 테마 설정
sns.set_theme(palette="Purples", style="whitegrid", font_scale=1) # 블루톤

# 그래프를 그리기 위한 기본 설정
plt.rcParams['font.family'] = 'Griun Gellyroll' # 제가... 고딕을 싫어해서요... (포폴용은 나눔스퀘어 씀)
# plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['figure.figsize'] = 12, 6
plt.rcParams['axes.titlesize'] = 16 # 제목 폰트 사이즈
plt.rcParams['axes.labelsize'] = 14 # 라벨 폰트 사이즈
plt.rcParams['font.size'] = 14 # 기본 폰트사이즈 
plt.rcParams['axes.unicode_minus'] = False

In [ ]:
# 데이터가 쓸만한게 있던가... 
df = pd.read_csv('data/Ibuprofen.csv', sep=';')

# 어차피 분석할거라 대충 채울게요 
df.fillna({'Targets':0.0}, inplace=True)
df.fillna({'Bioactivities':0}, inplace=True)
df.dropna(inplace=True)

df.shape

In [ ]:
# 수치형, 범주형 칼럼 분리

# 수치형 (정수, 실수 모두 포함)
numeric_columns = df.select_dtypes(include=['number']).columns

# 범주형 (문자열, Object 타입)
# categorical_columns = df.select_dtypes(include=['object']).columns <-이거 하면 이름같은거 다 껴서 안돼요 
# 착한 분석가 여러분들은 직접 확인하시고 픽하십셔
categorical_columns = ['Type', 'Passes Ro3', 'Structure Type']

## PCA
- sklearn.decomposition.PCA가 필요하다. 
- 데이터가 **수치형**일 경우에만 분석할 수 있기 떄문에 범주형은 더미화를 해야 하는데... 그냥 하지마세요... 결과 망해요. 내가 해봐서 아는거임. 

In [ ]:
# Scaler로 조정
scaler = StandardScaler()
numeric_scale = numeric_columns

transformed_numeric = scaler.fit_transform(df[numeric_scale]) # 변환된 데이터

transformed_numeric = pd.DataFrame(transformed_numeric, columns=numeric_scale) # 데이터프레임으로 변환

In [ ]:
pca = PCA(n_components=0.90)
X_pca = pca.fit_transform(transformed_numeric)

print("성공! 최종 변수 개수:", transformed_numeric.shape[1])
print("압축된 주성분 개수:", X_pca.shape[1])

- 저건 또 뭥미? 싶으시죠? 다 압니다. 근데 저 작업을 안 해주면 값이 주성분분석에 영향을 미쳐요. 
- 그 사태를 막기 위해 스탠다드 스케일러로 평균이 0, 표준편차가 1이 되도록 변환한다음 PCA를 진행한겁니다. 

### 결과 해석
- 위에서는 압축된 개수만 나오는데, 저거 사실 결과 볼거 더 있음... 

In [ ]:
# 1. 각 주성분의 설명력(분산 비율) 확인
explained_variance = pca.explained_variance_ratio_
cumulative_variance = np.cumsum(explained_variance)

print("--- PCA 결과 성적표 ---")
for i, ratio in enumerate(explained_variance):
    print(f"제 {i+1} 주성분: {ratio:.4f} (누적: {cumulative_variance[i]:.4f})")

# 2. 제1주성분(PC1)에 가장 큰 영향을 준 변수 top 10
# (주성분이 어떤 원래 변수들로 만들어졌는지 확인)
loadings = pd.DataFrame(pca.components_[0], index=numeric_columns, columns=['PC1_Loading'])
print("\n--- PC1을 구성하는 핵심 변수 TOP 10 ---")
print(loadings['PC1_Loading'].abs().sort_values(ascending=False).head(10))

1. PCA 결과 성적표
- 주성분분석은 원래 제 n 주성분 이런 식으로 변수명을 짓습니다. 그래서 변수명은 굳이 보실 필요 없고... 저 옆에 누적 숫자를 봐 주세요. 
- 제 6 주성분: 0.0547 (누적: 0.8564) <<이 누적 숫자가 의미하는 건 이 데이터가 85%의 설명력을 갖기 위해서 제 6 주성분까지 필요하다는 얘기입니다. 
- 이 데이터는 90% 이상의 설명력을 갖기 위해 제 8주성분까지 필요하네요. 

2. 제 1 주성분을 구성하는 핵심 변수 TOP 10
- 제 1 주성분이 이 변수들로 구성되어있다.. 이런 얘깁니다. 

### 시각화 (Scree plot)

In [ ]:
# 데이터 준비 (제공해주신 로직 유지)
pc_labels = []
for i in range(len(pca.components_)):
    # 최신 Pandas/NumPy 대응을 위해 numeric_scale.columns 사용 권장
    top_feature = numeric_columns[np.argmax(np.abs(pca.components_[i]))]
    pc_labels.append(f"PC{i+1}\n({top_feature})")

exp_var_pca = pca.explained_variance_ratio_

plt.figure(figsize=(16, 8))

# 1. 꺾은선 그래프 (Line + Marker)
sns.lineplot(x=pc_labels, y=exp_var_pca, marker='o', markersize=10, 
                color='purple', linewidth=2.5, label='Explained Variance')

# 2. 각 점 위에 수치 표시 (Annotate)
for i, val in enumerate(exp_var_pca):
    plt.text(i, val + 0.005, f'{val:.2f}', ha='center', va='bottom', fontsize=11)

plt.title('주성분별 설명력 및 대표 변수 (Scree Plot)', fontsize=15)
plt.ylabel('설명 가능한 분산 비율')
plt.xlabel('Principal Components')
plt.xticks(rotation=45)
plt.grid(axis='both', alpha=0.3) # 격자를 가로세로 모두 추가해서 흐름 파악 용이하게 함
plt.tight_layout()
plt.show()

- 씁... 원래 저게 완만해지는 부분(엘보)까지가 주성분이구나인데... 이거 결과랑 너무 다르잖아... 

### Loading plot

In [ ]:
# 로딩 값 추출
loadings = pca.components_.T * np.sqrt(pca.explained_variance_)
loading_df = pd.DataFrame(loadings, index=numeric_columns, columns=[f'PC{i+1}' for i in range(len(pca.components_))])

plt.figure(figsize=(10, 8))
plt.axhline(0, color='black', linewidth=1)
plt.axvline(0, color='black', linewidth=1)

# 주요 변수들만 화살표로 표시 (너무 많으면 복잡하므로 상위 기여 변수 위주)
top_features = loading_df.abs().sum(axis=1).sort_values(ascending=False).head(15).index

for feature in top_features:
    plt.arrow(0, 0, loading_df.loc[feature, 'PC1'], loading_df.loc[feature, 'PC2'], 
                color='r', alpha=0.5, head_width=0.02)
    plt.text(loading_df.loc[feature, 'PC1']*1.15, loading_df.loc[feature, 'PC2']*1.15, 
                feature, color='g', ha='center', va='center', fontsize=10)

plt.xlabel('PC1 (분자의 크기와 지질친화성)')
plt.ylabel('PC2 (극성 및 활성 지표)')
plt.title('변수별 로딩 플롯 (PC1 vs PC2)', fontsize=15)
plt.grid(alpha=0.3)
plt.xlim(-1, 1)
plt.ylim(-1, 1)
plt.tight_layout()
plt.show()

1. 저기서 X축은 제 1 주성분, Y축은 제 2 주성분을 나타내고 있습니다.
2. 화살표의 크기는 해당 변수가 주성분에 얼마나 영향을 미치는지를 나타냅니다.
3. 화살표의 방향은 해당 변수와 주성분 사이의 관계를 나타냅니다. 방향에 따라 무슨 관계인지가 다릅니다. 
    - 같은 방향(작은 각도): 변수들이 서로 양(+)의 상관관계를 가짐.
    - 반대 방향(180도): 변수들이 서로 음(-)의 상관관계를 가짐.
    - 90도 직교: 두 변수는 서로 상관관계가 거의 없음.

- 저어어어어어어어어어기 Heavy Atoms 보이시죠? 저게 가로축에 근접해 있습니다. 
    1. Heavy Atoms은 제 1 주성분에 양의 상관관계를 가지지만, 제 2 주성분(세로축)과는 거어어어어의 직교하고 있죠? 네. 극성이나 활성 지표랑은 1도 상관 없다는 의미입니다. 
    2. 반대로 Polar surface area는 제 2 주성분과 상관관계가 있지만, 제 1 주성분과는 아아아아아ㅏ아아아아아아무 상관이 없습니다. 
    3. 저기 Bioactivities는 3사분면에 있네요. 제 1 주성분, 제 2 주성분과 음의 상관관계를 갖는다는 얘기입니다. 
    

## MCA

In [ ]:
mca = prince.MCA(
    n_components=2,
    n_iter=3,
    copy=True,
    check_input=True,
    engine='sklearn',
    random_state=42
)

# 이친구는 데이터프레임만 취급합니다. 
mca = mca.fit(df[categorical_columns])

### 결과 확인

In [ ]:
mca_coords = mca.row_coordinates(df[categorical_columns])

print(f'Coordinates of the rows (MCA): {len(mca_coords)}')
mca_coords

- 악 내눈! 
- 우리 데이터프레임에서 결측값 다 쳐내고 59개의 화합물이 남았어요. (직접 확인해봤음). 그리고 범주형 나눈 게 12개거든요? 그러니까 저건 
    1. 59개의 화합물에 대해서 
    2. 12개의 범주형 칼럼들을 다 매겨서
    3. 얘네들끼리 얼마나 비슷한지 좌표를 나타냈음

    이런겁니다. 그래서 저것만 봐서는 이게 뭔지 모르는 게 정상이예요. 

In [ ]:
# 1. 각 카테고리의 차원별 기여도 확인
# 최신 prince 버전에서는 'column_contributions_' 속성을 사용합니다.
col_contributions = mca.column_contributions_

# 2. 0번 차원(축)을 만드는 데 가장 크게 기여한 상위 5개 항목
print("--- 0번 축(Dimension 0)의 정체 ---")
print(col_contributions[0].sort_values(ascending=False).head(5))

print("\n--- 1번 축(Dimension 1)의 정체 ---")
print(col_contributions[1].sort_values(ascending=False).head(5))

### 시각화

In [ ]:
plt.figure(figsize=(10, 8))

# 좌표 데이터 산점도 그리기
sns.scatterplot(x=mca_coords[0], y=mca_coords[1], alpha=0.6, color='royalblue')

# 중심선 그리기 (원점 기준)
plt.axhline(0, color='gray', linestyle='--', linewidth=1)
plt.axvline(0, color='gray', linestyle='--', linewidth=1)

plt.title('MCA 결과: 화합물 간의 범주적 유사도 지도', fontsize=15)
plt.xlabel('MCA 축 1', fontsize=12)
plt.ylabel('MCA 축 2', fontsize=12)
plt.grid(alpha=0.2)
plt.show()

- 근데 시각화 해도 모르는게 정상이냐고... 

In [ ]:
plt.figure(figsize=(10, 8))
# 점들이 겹치지 않게 살짝 흩뿌려주는 jitter 효과를 줍니다.
sns.stripplot(x=mca_coords[0], y=mca_coords[1], alpha=0.6, jitter=True, color='purple')
plt.title('MCA 결과: 겹친 점들을 펼쳐서 보기', fontsize=15)
plt.show()

## FAMD
- 이친구는 수치형, 범주형 안 가립니다. 

In [ ]:
# 1. 인덱스 때문에 꼬이는 경우가 많으니 복사본 생성 후 인덱스 초기화
famd_input = df[list(numeric_columns) + list(categorical_columns)].copy().reset_index(drop=True)

# 2. 혹시 모르니 모든 수치형 칼럼을 float로 강제 변환
for col in numeric_columns:
    famd_input[col] = famd_input[col].astype(float)

# 3. FAMD 다시 실행
famd = prince.FAMD(
    n_components=2,
    n_iter=5, # 반복 횟수를 조금 늘려주면 안정적입니다
    engine='sklearn',
    random_state=42
)

# 피팅 시도
try:
    famd = famd.fit(famd_input)
    print("성공! 이제 좌표를 뽑을 수 있습니다.")
except ValueError as e:
    print(f"아직 에러가 나네요: {e}")
    # 만약 또 에러가 나면, 범주형 칼럼이 너무 적어서(3개) 발생할 수 있습니다.

### 결과 확인

In [ ]:
famd_coords = famd.row_coordinates(famd_input)

famd_coords

In [ ]:
# 1. 설명력 (%) - 소수점 첫째 자리까지 + '%' 붙이기
explain_rounded = [f"{v:.1f}%" for v in famd.percentage_of_variance_]

# 2. 원점수 (고윳값) - 소수점 셋째 자리까지 깔끔하게
point_rounded = [round(v, 3) for v in famd.eigenvalues_]

print(f'해당 차원의 설명력: {explain_rounded}')
print(f'해당 차원의 원점수: {point_rounded}')

- 악 내눈! ㅋㅋㅋㅋㅋㅋ 위에 있는 건 MCA에서도 봤던 좌표입니다. 그리고 아래는 차원 두 개의 설명력이고요. 

### 각 차원별 기여도 순위
#### 0차원

In [ ]:
# 각 칼럼들이 0차원에 기여하는 순서대로 정렬
df_comp0 = famd.column_contributions_.sort_values(by=0, ascending=False)

# 해서 Top 10만 
df_comp0[[0]].head(10)

#### 1차원

In [ ]:
# 각 칼럼들이 1차원에 기여하는 순서대로 정렬
df_comp1 = famd.column_contributions_.sort_values(by=1, ascending=False)

# Top 10
df_comp1[[1]].head(10)

### 차원 기여도 시각화

In [ ]:
# 0번 차원 기여도 시각화
# 0차원: "Molecular Complexity & Size"
famd.column_contributions_[0].sort_values().plot(kind='barh', figsize=(8, 6), color='#00498c')
plt.title('Component 0에 대한 변수 기여도')
plt.xlabel('Contribution (%)')
plt.show()

In [ ]:
# 1번 차원 기여도 시각화
# 1차원: "Polarity & Activity Profile"
famd.column_contributions_[1].sort_values().plot(kind='barh', figsize=(8, 6), color='#00498c')
plt.title('Component 1에 대한 변수 기여도')
plt.xlabel('Contribution (%)')
plt.show()

#### 한짤요약

In [ ]:
plt.figure(figsize = (20, 10))

plt.subplot(1, 2, 1)
famd.column_contributions_[0].sort_values().plot(kind='barh', figsize=(8, 6), color='#00498c')
plt.title('Component 0에 대한 변수 기여도')
plt.xlabel('Contribution (%)')

plt.subplot(1, 2, 2)
famd.column_contributions_[1].sort_values().plot(kind='barh', figsize=(8, 6), color='#00498c')
plt.title('Component 1에 대한 변수 기여도')
plt.xlabel('Contribution (%)')
plt.tight_layout()
plt.show()

- 근데 니네 뭐 하는 차원임? 이 궁금하시죠? 그건 저걸 보고 우리가 유추해야 합니다... ㅋㅋㅋㅋㅋㅋ 
- 저 차원을 구성하는 주요 요소들을 통해 유추하시면 됩니다. 

### Scatter plot

In [ ]:
# 나눔스퀘어 폰트는 이미 설정되어 있으니 바로 사용합니다.
plt.figure(figsize=(12, 9))

# 1. 산점도 그리기
# 색상은 'Type'(Small Molecule 등), 모양은 'Passes Ro3'(약물성 통과 여부)로 구분
scatter = sns.scatterplot(
    x=famd_coords[0], 
    y=famd_coords[1], 
    style=df['Type'], 
    hue=df['Molecular Weight'],
    s=150,           # 점 크기 확대
    alpha=0.7,       # 투명도 조절로 겹친 부분 확인
    edgecolor='w',   # 점 테두리 흰색으로 구분감 상승
    palette='viridis' # 세련된 색감 적용
)

# 2. 축 이름 지어주기 (기여도 분석 결과 반영)
# 설명력(percentage_of_variance_)을 소수점 첫째 자리까지 포함
plt.xlabel(f'Dimension 0: 화합물의 체급 및 복잡도 ({famd.percentage_of_variance_[0]:.1f}%)', fontsize=12)
plt.ylabel(f'Dimension 1: 극성 및 생물학적 활성 프로파일 ({famd.percentage_of_variance_[1]:.1f}%)', fontsize=12)

# 3. 보조 가이드라인 (원점 기준)
plt.axhline(0, color='gray', linestyle='--', alpha=0.3)
plt.axvline(0, color='gray', linestyle='--', alpha=0.3)

# 4. 제목 및 범례 설정
plt.title('FAMD 최종 분석: ChEMBL 화합물의 다차원 지형도', fontsize=16, pad=20)
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', title='변수 구분')

plt.grid(True, linestyle=':', alpha=0.4)
plt.tight_layout()
plt.show()

#### 쟤 뭔데 혼자 노냐

In [ ]:
# FAMD 좌표에서 0번 축 값이 가장 작거나 큰 (끝단에 있는) 인덱스 찾기
outlier_idx = famd_coords[0].idxmin() # 혹은 idxmax() 위치에 따라 선택
print(f"저 외로운 녀석의 이름: {df.loc[outlier_idx, 'Name']}")

In [ ]:
# 이부프로펜과 알미노프로펜 비교 (이름이 정확한지 확인 필요)
comparison = df[df['Name'].isin(['ALMINOPROFEN', 'IBUPROFEN'])][['Name', 'Heavy Atoms', 'Molecular Weight', 'Polar Surface Area', 'AlogP']]
print(comparison)

In [ ]:
# 1. 0번 차원이 가장 작고, 1번 차원이 5 근처인 행 찾기
# (절댓값이 아니라 실제 '가장 작은 값' 기준)
the_one = famd_coords[(famd_coords[0] < -4) & (famd_coords[1] > 5)]

if not the_one.empty:
    idx = the_one.index[0]
    print(f"--- 최종 확인된 주인공 ---")
    print(f"이름: {df.loc[idx, 'Name']}")
    print(f"좌표: Dim0({famd_coords.loc[idx, 0]:.2f}), Dim1({famd_coords.loc[idx, 1]:.2f})")
    print(f"PSA: {df.loc[idx, 'Polar Surface Area']}, MW: {df.loc[idx, 'Molecular Weight']}")
else:
    print("해당 좌표 근처에 아무도 없어요. 다시 한 번 좌표를 확인해볼까요?")

- 분자량은 작은데 극성 면적이 꽤 크네요. 